In [15]:
import numpy as np
import pandas as pd

import tabularepimdl as tepi

from tabularepimdl.SimpleTransition_Vec_Encode import SimpleTransition_Vec_Encode
from tabularepimdl.SimpleInfection_Vec_Encode import SimpleInfection_Vec_Encode

from tabularepimdl.MultiStrainInfectiousProcess_Vec_Encode import MultiStrainInfectiousProcess_Vec_Encode

from tabularepimdl.EpiModel_Vec_Encode1_2 import EpiModel_Vec_Encode_1_2
#from tabularepimdl.EpiModel_orig import EpiModel_orig

import plotly.express as px
import plotly.graph_objects as go

In [2]:
infstate_compartments = ['S', 'I', 'R']
simpletrans_column_compartments =  ['S', 'I', 'R']
multistrain_columns_all_categories = ['S', 'I', 'R']

#define immunity such tht "adjacent" strains have 50% cross protection and "non-adjacent" strains have 20% cross protection.
cp_matrix = np.array([[1.0, 0.5, 0.2],
                      [0.5, 1.0, 0.5],
                      [0.2, 0.5, 1.0]])

pop_3strain = pd.DataFrame(
    {
        "Strain1": pd.Categorical(['S','I'], categories=['S','I','R']),
        "Strain2": pd.Categorical(['S','S'], categories=['S','I','R']),
        "Strain3": pd.Categorical(['S','S'], categories=['S','I','R']),
        "N": np.array([990,10]),
        "T":0
    }
)

In [3]:
#test_pop = pd.DataFrame(
#    {
#        "Strain1": pd.Categorical(['S'], categories=['S','I','R']),
#        "Strain2": pd.Categorical(['S'], categories=['S','I','R']),
#        "Strain3": pd.Categorical(['S'], categories=['S','I','R']),
#        "N": np.array([10]),
#        "T":0
#    }
#)
#test_pop

In [4]:
three_strain_infect_vec = MultiStrainInfectiousProcess_Vec_Encode(betas=np.array([0.5, 0.75, 1]), columns=["Strain1", "Strain2", "Strain3"], cross_protect=cp_matrix, columns_all_categories = multistrain_columns_all_categories, infstate_compartments=infstate_compartments)

In [5]:
recover_rule1_vec = SimpleTransition_Vec_Encode(column='Strain1', from_st='I', to_st='R', rate=0.2, infstate_compartments=infstate_compartments, column_categories=simpletrans_column_compartments)
recover_rule2_vec = SimpleTransition_Vec_Encode(column='Strain2', from_st='I', to_st='R', rate=0.2, infstate_compartments=infstate_compartments, column_categories=simpletrans_column_compartments)
recover_rule3_vec = SimpleTransition_Vec_Encode(column='Strain3', from_st='I', to_st='R', rate=0.2, infstate_compartments=infstate_compartments, column_categories=simpletrans_column_compartments)

In [6]:
#SIR_3Strain_vec = EpiModel_Vec_Encode_1_2(init_state=test_pop, rules = [three_strain_infect_vec, recover_rule1_vec, recover_rule2_vec, recover_rule3_vec]) #test
SIR_3Strain_vec = EpiModel_Vec_Encode_1_2(init_state=pop_3strain, rules = [three_strain_infect_vec, recover_rule1_vec, recover_rule2_vec, recover_rule3_vec])

In [7]:
SIR_3Strain_vec._col_idx_map,\
three_strain_infect_vec._columns_idx

({'Strain1': 0, 'Strain2': 1, 'Strain3': 2, 'N': 3, 'T': 4}, [])

In [8]:
three_strain_infect_vec._s_code, three_strain_infect_vec._i_code, three_strain_infect_vec._r_code, three_strain_infect_vec._inf_to_code

(2, 0, 1, 0)

In [9]:
#intro_day = 0.5
#for t in np.arange(0, 1, 0.25):
#    print('t --- :', t)
#    if t == intro_day:
#        to_add = pd.DataFrame({ #on 11th day, there is one person infected with strain2 and susceptiable to strain1 and 3
#            'Strain1':pd.Categorical(['S'], categories=['S','I','R']),
#            'Strain2':pd.Categorical(['I'], categories=['S','I','R']),
#            'Strain3':pd.Categorical(['S'], categories=['S','I','R']),
#            'N':1,
#            'T':[t]
#         })
#        SIR_3Strain_vec.add_new_data_to_current_state(new_data = to_add)
#        print('add new data, now vec cur state\n', SIR_3Strain_vec.current_state_array)
#    SIR_3Strain_vec.do_timestep(dt=0.25)

In [10]:
intro_day1 = 11
intro_day2 = 20
for t in np.arange(0, 100, 0.25):
    #print('t --- :', t)
    if t == intro_day1:
        to_add = pd.DataFrame({ #on 11th day, there is one person infected with strain2 and susceptiable to strain1 and 3
            'Strain1':pd.Categorical(['S'], categories=['S','I','R']),
            'Strain2':pd.Categorical(['I'], categories=['S','I','R']),
            'Strain3':pd.Categorical(['S'], categories=['S','I','R']),
            'N':1,
            'T':[t]
         })
        SIR_3Strain_vec.add_new_data_to_current_state(new_data = to_add)
        #print('add new data, now vec cur state\n', SIR_3Strain_vec.current_state_array)
    if t == intro_day2:
        to_add = pd.DataFrame({ #on 20th day, there is one person infected with strain3 and susceptiable to strain1 and 2
            'Strain1':pd.Categorical(['S'], categories=['S','I','R']),
            'Strain2':pd.Categorical(['S'], categories=['S','I','R']),
            'Strain3':pd.Categorical(['I'], categories=['S','I','R']),
            'N':1,
            'T':[t]
         })
        SIR_3Strain_vec.add_new_data_to_current_state(new_data = to_add)
    SIR_3Strain_vec.do_timestep(dt=0.25)

In [12]:
#SIR_3Strain_vec.current_state_array

In [13]:
SIR_3Strain_vec_full_epi = SIR_3Strain_vec._covnert_list_of_arrays_to_df(SIR_3Strain_vec._full_epi_list)

In [16]:
long_epi_determ = SIR_3Strain_vec_full_epi.melt(id_vars=['N','T'], value_vars=['Strain1', 'Strain2', 'Strain3'], var_name='Strain', value_name="InfState")
long_epi_determ = long_epi_determ.groupby(["T","Strain","InfState"]).sum().reset_index()

epi_fig_determ = px.line(long_epi_determ, x="T", y="N", color="InfState", line_dash="Strain", title='3 Strains Infection Change on a Population - Deterministic')
epi_fig_determ.show()

#### Making a Stochastic Version
Reset the model and set the EpiModel.stoch_policy parameter to "stochastic". We will do multiple runs where the introduction day is a random variable.

In [17]:
#Making a Stochastic Version
SIR_3Strain_vec.Reset()
SIR_3Strain_vec.stoch_policy="stochastic"

##go through multiple times appending the simulation. 
all_sims = pd.DataFrame() #create an empty dataframe

for sim in range(15):
    #print('sim: ', sim) #debug
    SIR_3Strain_vec.Reset()
    
    intro_day1 = np.random.poisson(10)
    intro_day2 = intro_day1 + np.random.poisson(10)
    #print('intro_day1 is {}, intro_day2 is {}'.format(intro_day1, intro_day2)) #debug
    for t in np.arange(0, 100, 0.25):
        #print('t is: ', t) #debug
        if t == intro_day1:
            to_add = pd.DataFrame({
                'Strain1':pd.Categorical(['S'], categories=['S','I','R']),
                'Strain2':pd.Categorical(['I'], categories=['S','I','R']),
                'Strain3':pd.Categorical(['S'], categories=['S','I','R']),
                'N':1,
                'T':[t]
            }) 
            SIR_3Strain_vec.add_new_data_to_current_state(new_data = to_add)
        if t == intro_day2:
            to_add = pd.DataFrame({
                'Strain1':pd.Categorical(['S'], categories=['S','I','R']),
                'Strain2':pd.Categorical(['S'], categories=['S','I','R']),
                'Strain3':pd.Categorical(['I'], categories=['S','I','R']),
                'N':1,
                'T':[t]
            })
            SIR_3Strain_vec.add_new_data_to_current_state(new_data = to_add)
        SIR_3Strain_vec.do_timestep(dt=0.25)

    tmp = SIR_3Strain_vec._covnert_list_of_arrays_to_df(SIR_3Strain_vec._full_epi_list) #convert each sim's full_epi to dataframe
    tmp['sim'] = sim
    all_sims = pd.concat([all_sims,tmp])

all_sims.reset_index(drop=True)

,Strain1,Strain2,Strain3,N,T,sim
0,S,S,S,990.0,0.00,0
1,I,S,S,10.0,0.00,0
2,I,S,S,9.0,0.25,0
3,R,S,S,2.0,0.25,0
4,S,S,S,989.0,0.25,0
...,...,...,...,...,...,...
39864,R,S,R,242.0,100.00,14
39865,R,S,S,2.0,100.00,14
39866,S,R,R,85.0,100.00,14
39867,S,R,S,12.0,100.00,14


In [18]:
long_epi_stoch = all_sims.melt(id_vars=['N','T','sim'], value_vars=['Strain1', 'Strain2', 'Strain3'], var_name='Strain', value_name="InfState")
long_epi_stoch = long_epi_stoch.groupby(["T","Strain","InfState","sim"]).sum().reset_index()

epi_fig_stoch = px.line(long_epi_stoch, x="T", y="N", color="InfState", line_dash="Strain", line_group="sim", title='3 Strains Infection Change on a Population - Stochastic')
epi_fig_stoch.update_traces(opacity=0.25)
epi_fig_stoch.show()